# ***PDF to RAG pipeline***

#### ***Notebook Sections (Cell Mapping)***
1. **Imports & Environment**
    - `Package initialization`
    - `configurations`
    - `Standard data/ML libraries`
    - `Logger configuration`
    - Validation: Confirm CONFIG_PATH exists and config is valid YAML dict 
2. **Utilities Functions  & Text Pre-Processing**
    - `normalize_whitespace`
    - `remove_control_chars`
    - `dedupe_lines`
    - `format_llm_output`
3. **PDF Extraction & Text Cleaning** 
    - `extract_text`
    - `clean_text`
    - `remove_title_section`
    - `remove_ending_section`
    - `save_clean_text`
4. **Tokenization & Chunking**
    - `tokens_of`
    - `detokenize`
    - `extract_key_terms` 
    - `chunk_text`
    - `refine_chunks` `(cl100k_base, 125 tok, 0.20 overlap, 50-700 refined)`
5. **Embeddings & FAISS Index Construction**
    - `build_embeddings_and_index` `(SentenceTransformer all-MiniLM-L6-v2)`
    - `cosine similarity via inner-product`
    - `index persistence` 
6. **Embedding Statistics & Analysis**
    - `Visual: Interactive Embedding Space Projection`
    - `calculate_l2_stats`
    - `sample_pairwise_l2_distances`
    - `calculate_cosine_similarity_stats`
7. **Domain Ontology & Knowledge Graph Construction**
    - `DomainOntology class`
       - ` _load_expanded_vocabulary`
       - `extract_entities_from_text`
    - `KnowledgeGraph class`
       - `add_chunk_to_graph`
       - ` _extract_keywords`
       - `find_related_chunks` 
8. **Semantic Layer & Domain Reasoning**
    - `SemanticLayer class`
       - ` ground_query` 
         - `entity type detection` 
         - `intent classification (synthesis / enumeration / explanatory)`
9.  **Hybrid Retrieval Orchestration**
    - `retrieve_hybrid` (Vector + Graph + Keyword with alpha weighting), Multi-modal fusion logic
    - `synthesize_answer_from_chunks`  (ranked result assembly)
10. **Retriever & Index Loading**
    - `retrieve`
    - `assemble_context`  
11. **LLM Setup & Prompting**
     - `LLMWrapper class`
     - `llm = LLMWrapper` (initializer)
     - `call_llm` (t5-base, prompt engineering with context constraint 374 tokens, citation formatting)
12.  **Enhanced RAG Answer Generation**
     - `enhanced_rag_answer` (semantic grounding → hybrid retrieval → generation → fallback synthesis) (Cross_Encoder)
13.  **Post-Processing, Citation Extraction & Cleanup**
     - `extract_and_remove_citations`  ( Parses inline [chunk_id] markers, deduplicates, returns clean text + indices)
     - `post_process_enhanced` (remove inline citations, fix proper nouns, normalize whitespace, extract citation indices also handles semantic_grounding extraction)
14. **MAIN EXECUTION: Knowledge Graph Initialization & Population**
     - `Initialize: DomainOntology(), KnowledgeGraph(), SemanticLayer()`
     - `Populate graph via kg.add_chunk_to_graph()`
15.  **Core Thesis Query Execution**
     - Execute Query 1 with semantic grounding output, save to `article_answer.txt` 
16.  **Targeted Follow-Up Questions**
     - Generate + answer 5 domain questions iteratively across metrics, geography, barriers, strategy, hybrid retrieval per question, save to `article_qa_pairs.txt`
17.  **Comprehensive Summary Generation**
     - Synthesize cross-chunk insights, enhanced with fallback synthesis logic, save to `article_summary.txt`
19.  **Test**
     - len(final_answer) > 40 characters 
     - leader' in final_answer.lower() 
20.  **Analysis Report & Knowledge Graph Export**
     - Generate JSON report (entities, relationships, retrieval stats), export `knowledge_graph_entities.txt`
21.  **Pipeline Summary & Final Validation**
     - Display entity count, chunks processed, retrieval method, output format, question count, summary length
     - Test Validation Checks:
         * Structural: Verify Q&A count (5/5) and summary sentence range (5-8)
         * Grounding: Confirm all citations reference real source chunks (prevents hallucination)
         * Persistence: Validate all 5 output files exist and are non-empty
         * Data Integrity: Verify JSON report is valid object structure
     - Output: Detailed pass/fail report with verification breakdown
     - **Test Call:** `from test_integration import run_validation; run_validation(qa_results, summary_text, chunks, output_dir)`


`NOTE:`Integration Test Suite**
        - Run `test_integration.py` against the generated artifacts
        - Validates end-to-end pipeline correctness
        - (Can be run separately or embedded as final cell) 

### ***Import & Environment*** 

In [2]:
# Standard Library 
import json
import os
import re
import sys
import textwrap
import time
import traceback
import warnings
from datetime import datetime
import pathlib
from pathlib import Path
from typing import cast, Any, Optional 

# Data Handling & Scientific Computing 
import numpy as np
import pandas as pd
from numpy.typing import NDArray
from sklearn.metrics.pairwise import cosine_similarity

# PDF Processing 
import pdfplumber

# Deep Learning & LLM Frameworks
import tensorflow as tf
import tiktoken
import torch
import yaml
from sentence_transformers import CrossEncoder, SentenceTransformer
from transformers import T5ForConditionalGeneration, T5Tokenizer, AutoTokenizer, AutoModelForSeq2SeqLM

# Vector Store & Similarity Search 
import faiss
from faiss import IndexFlatL2

# Visualization 
import matplotlib.pyplot as plt
import plotly.graph_objects as go

# Jupyter Configuration 
%matplotlib inline

# System Configuration & Warning Suppression
# Suppress TensorFlow and Deprecation warnings for cleaner notebook output  
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # 0=all, 1=INFO, 2=WARNING, 3=ERROR
warnings.filterwarnings('ignore', message='.*sparse_softmax_cross_entropy.*')
warnings.filterwarnings("ignore", category=DeprecationWarning)
tf.get_logger().setLevel("ERROR")

In [ ]:
# Resolve parent directory to allow importing from local project modules (e.g., logs.logger)
sys.path.append(str(Path("..").resolve()))

# Local Project Imports 
from logs.logger import get_logger

# Initialize centralized logging for pipeline traceability
log = get_logger()
log.info("Pipeline started")

# Define project root and configuration path for relative resource access
PROJECT_ROOT = pathlib.Path("..").resolve()
CONFIG_PATH = pathlib.Path("../configs/config.yaml").resolve()   

# Ensure configuration file exists before attempting to load
if not CONFIG_PATH.is_file():
    raise FileNotFoundError(f"Configuration file not found: {CONFIG_PATH}")

# Load YAML configuration and validate that the root structure is a dictionary
_cfg_raw = yaml.safe_load(CONFIG_PATH.read_text())
if not isinstance(_cfg_raw, dict):
    raise TypeError(
        f"The config file must contain a YAML mapping (dictionary). Got: {type(_cfg_raw)}"
    )
_cfg = _cfg_raw   

# Text segmentation hyperparameters for optimized context windows
CHUNK_SIZE    = _cfg["chunking"]["size"]          
OVERLAP_RATIO = _cfg["chunking"]["overlap"]       

# Hybrid retrieval constraints for balancing vector and graph recall
TOP_K = _cfg["retrieval"]["top_k"]
SCORE_THRESHOLD = _cfg["retrieval"]["score_threshold"]

# Generation settings for the local T5 model to control output length and creativity
MAX_NEW_TOKENS = _cfg["llm"]["max_new_tokens"]
TEMPERATURE    = _cfg["llm"]["temperature"]
SYSTEM_PROMPT  = _cfg["llm"]["system_prompt"]

In [ ]:
print("TOP_K =", TOP_K)
print("LLM will generate up to", MAX_NEW_TOKENS, "tokens")

### ***Utilities Functions  & Text Pre-Processing***

In [ ]:
# UTILITY FUNCTIONS     
def normalize_whitespace(s: str) -> str:
    """
    Normalize various whitespace patterns in text.

    Performs the following transformations:
    - Converts CRLF and CR line endings to LF
    - Collapses consecutive spaces and tabs to single space
    - Reduces 3+ consecutive newlines to exactly 2 newlines
    - Strips leading and trailing whitespace

    Args:
        s (str): Input text to normalize.

    Returns:
        str: Normalized text, or original input if an error occurs.
    """
    try:
        s = re.sub(r'\r\n?', '\n', s)  # Normalize line endings to LF
        s = re.sub(r'[ \t]+', ' ', s)  # Collapse whitespace runs
        s = re.sub(r'\n{3,}', '\n\n', s) # Limit consecutive newlines
        return s.strip()
    except Exception as e:
        print(f"Error in normalize_whitespace: {e}")
        return s  

def remove_control_chars(s: str) -> str:   
    """
    Remove C0 control characters while preserving text content.

    Removes all ASCII control characters (0x00–0x1F, 0x7F) except newline (0x0A/LF),
    which is retained to preserve document structure.

    Args:
        s (str): Input text.

    Returns:
        str: Text with control characters removed, or original input if an error occurs.
    """ 
    try:
        return re.sub(r'[\x00-\x09\x0B-\x1F\x7F]', '', s)
    except Exception as e:
        print(f"Error in remove_control_chars: {e}")
        return s  

def dedupe_lines(s: str) -> str:           
    """
    Remove duplicate lines while preserving order of first occurrence.

    Skips empty or whitespace-only lines entirely and removes any subsequent
    occurrence of a line already seen.

    Args:
        s (str): Input text with newline-separated lines.

    Returns:
        str: Deduplicated text, or original input if an error occurs.
    """
    try:
        lines = []
        seen = set()
        for ln in s.splitlines():
            t = ln.strip()
            # Skip empty lines and skip duplicates (track with set for O(1) lookup)
            if not t:
                continue
            if t in seen: 
                continue
            seen.add(t); lines.append(t)
        return '\n'.join(lines)
    except Exception as e:
        print(f"Error in dedupe_lines: {e}")
        return s
    
def format_llm_output(text: str) -> str:
    """
    Fixes common LLM formatting errors: 
    - Removes spaces before punctuation
    - Ensures sentences start with capital letters
    """
    if not text:
        return ""
    
    # Remove spaces before punctuation (e.g., "landscape ." -> "landscape.")
    text = re.sub(r'\s+([\.!\?,;:])', r'\1', text)
    
    # Fix capitalization: Capitalize the first letter of every sentence
    text = re.sub(r'(^|[.!?]\s+)([a-z])', lambda m: m.group(1) + m.group(2).upper(), text)
    
    # Final whitespace cleanup
    text = ' '.join(text.split())
    
    return text  

### ***PDF Extraction & Text Cleaning***

In [ ]:
def extract_text(pdf_path: str) -> str:    
    """
    Extract text from all pages of a PDF and wrap each with page markers.

    Args:
        pdf_path (str): Path to the PDF file.

    Returns:
        str: Concatenated text from all pages with [PAGE_N_START] and [PAGE_N_END] markers,
             or the original pdf_path string if extraction fails.
    """ 
    try:
        full = []
        with pdfplumber.open(pdf_path) as pdf:
            # Extract text from each page and wrap with identifiable markers for downstream processing
            for i, page in enumerate(pdf.pages, start=1):
                text = page.extract_text() or ""
                if text:
                    full.append(f"[PAGE_{i}_START]\n" + text + f"\n[PAGE_{i}_END]") 
        return "\n\n".join(full)
    except Exception as e:
        print(f"Error in extract_text for {pdf_path}: {e}") 
        return pdf_path  

In [ ]:
def clean_text(raw: str) -> str:     
    """
    Comprehensively clean and normalize extracted text.

    Applies a sequence of transformations: removes control characters, normalizes
    whitespace, deduplicates lines, fixes line-break artifacts, and applies
    heuristics for sentence and word boundary detection.

    Args:
        raw (str): Raw extracted text from PDF or other source.

    Returns:
        str: Cleaned and normalized text, or original input if an error occurs.
    """ 
    try:
        s = remove_control_chars(raw)
        s = normalize_whitespace(s)
        s = dedupe_lines(s)
        
         # Remove hyphens at line breaks to rejoin hyphenated words across lines
        s = s.replace("-\n", "") 
        # Convert lines with excess trailing whitespace to paragraph breaks 
        s = re.sub(r'[ \t]{2,}\n', '\n\n', s) 
        # Join single newlines with spaces (treat as line-wrap continuation, not paragraph break) 
        s = re.sub(r'(?<!\n)\n(?!\n)', ' ', s)
        # Collapse multiple spaces into single space  
        s = re.sub(r' {2,}', ' ', s)
        # Add space between lowercase/digit and uppercase letter (e.g., "exampleText" → "example Text")  
        s = re.sub(r'(?<=[a-z0-9\)\]\-,:;])(?=[A-Z])', ' ', s)
        # Collapse multiple newlines into exactly two  
        s = re.sub(r'\n{3,}', '\n\n', s)
        # Remove newlines from very long lines (likely OCR artifacts from wrapped text) 
        s = re.sub(r'([^\n]{1000,}?)\n', lambda m: m.group(1) + ' ', s)
        # Placeholder for potential future merged-word handling (currently no-op)
        s = re.sub(r'(?<=\w)(?=\w)', '', s)  
        return s.strip()
    except Exception as e:
        print(f"Error in clean_text: {e}")
        return raw 